# Protocolo três juízes

Roda o protocolo inteiro no Colab com GPU. O `.env` com o `HF_TOKEN` fica na pasta `IniciacaoCientifica/colab` do Drive, e as saídas e o BERTimbau treinado voltam para lá.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

pasta = "/content/drive/MyDrive/IniciacaoCientifica/colab"

In [ ]:
!git clone -b staging https://github.com/zerikazz/transformers-project.git /content/transformers-project
%cd /content/transformers-project
!pip install -q -r requirements.txt bitsandbytes

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(f"{pasta}/.env")
os.environ["AMOSTRA"] = "600"

## Dados

In [ ]:
!python -m preparar.carolina
!python -m preparar.tweets
!python -m preparar.goemotions
!python -m protocolo.lexico

## Polaridade, curvas e categórico x contínuo

In [ ]:
!python -m experimentos.finetune_bertimbau
!ENTRADA=dados/tweets_test.csv SAIDA=saida/polaridade_tweets.csv AMOSTRA=870 python -m protocolo.polaridade
!AMOSTRA=500 python -m protocolo.polaridade
!python -m experimentos.curvas
!python -m experimentos.categorico

## Três juízes e a hipótese

In [ ]:
!FAMILIA=transformer CONJUNTO=goemotions python -m protocolo.protocolo
!FAMILIA=deepseek CONJUNTO=goemotions python -m protocolo.protocolo
!FAMILIA=deepseek CONJUNTO=goemotions LLM=deepseek-ai/DeepSeek-R1-Distill-Qwen-7B QUANTIZAR=4bit python -m protocolo.protocolo

In [ ]:
!mkdir -p "{pasta}/saida"
!cp -r saida/* "{pasta}/saida/"